# 06-01 优化器进阶：RMSProp、Adam 与 AdamW

前面我们已经学过指数加权移动平均：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

现在可以重新理解优化器。

SGD 只看当前梯度，Momentum 用 EMA 记住梯度方向，RMSProp 用 EMA 记住梯度大小，Adam 则把这两个思想合在一起。

## 1. 为什么 SGD 还不够

SGD 的更新公式是：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

其中：

$$
g_t=\nabla_{\theta}\mathcal{L}_t
$$

SGD 的问题是：所有参数共用同一个学习率 $\eta$。

但不同参数的梯度情况可能完全不同。

有的参数梯度经常很大，说明它变化很敏感；如果还用同样的大步子，可能震荡。

有的参数梯度经常很小，说明它更新很慢；如果还用同样的小步子，可能几乎不动。

所以问题变成：**能不能给不同参数自动调整不同的实际步幅？**

## 2. 自适应学习率先别想复杂

先把学习率想成“每次改参数时迈多大一步”。

SGD 的做法比较直接：所有参数都听同一个命令。

```text
大家都走 η 这么大一步
```

但神经网络里的参数不是同一种情况。

有些参数经常被数据影响，梯度经常很大。它们像是已经被反复提醒的参数，如果还让它大步走，就容易来回晃。

有些参数很少被数据影响，梯度经常很小。它们像是很少有机会被纠正的参数，如果还只让它小步走，就可能学得特别慢。

所以自适应学习率想解决的是一句很朴素的话：

```text
不要所有参数都走同样大的步子。
```

更具体一点：

```text
经常变化很大的参数：以后谨慎一点，步子小一点
很少变化或变化较小的参数：不要压得太死，可以相对多走一点
```

AdaGrad、RMSProp、Adam 都是在围绕这件事做文章。区别只是：它们用什么方式判断“这个参数过去变化大不大”。

## 3. AdaGrad 先解决了什么

AdaGrad 的名字可以拆开看：

```text
Ada = Adaptive，自适应
Grad = Gradient，梯度
```

也就是：**根据每个参数自己的梯度情况，自动调整它的实际步子。**

它的核心做法很像给每个参数单独记一本账：

```text
这个参数过去梯度大不大？
如果过去经常很大，就说明它很敏感，以后少走点。
如果过去没怎么大过，就不要把它的步子压得太小。
```

怎么记账？

假设当前第 $t$ 次更新时，某个参数的梯度是：

$$
g_t
$$

AdaGrad 不直接只看这一次的 $g_t$，它会把这个参数过去每一次梯度的平方累加起来：

$$
s_t=s_{t-1}+g_t^2
$$

这个 $s_t$ 可以先别叫“二阶矩”这种名字，初学阶段你就把它理解成：

```text
这个参数到目前为止，梯度总共“激烈”过多少次。
```

为什么要平方？

第一，梯度有正有负，平方以后都变成正数，方便统计“大小”。

第二，较大的梯度平方后会更明显。例如：

$$
2^2=4,\qquad 10^2=100
$$

所以 $s_t$ 越大，就说明这个参数历史上越“活跃”、越“敏感”。

接着 AdaGrad 更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

这个公式不要一整坨看。它其实还是 SGD：

$$
\theta_t=\theta_{t-1}-\text{步子}
$$

只不过 AdaGrad 把原来的步子：

$$
\eta g_t
$$

改成了：

$$
\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

也就是用 $\frac{1}{\sqrt{s_t}+\epsilon}$ 给当前梯度踩了一脚刹车。

如果 $s_t$ 很大，分母大，刹车重，实际更新就小。

如果 $s_t$ 不大，分母没那么大，刹车轻，实际更新就相对大。

这就是 AdaGrad 的主线：

```text
给每个参数记历史梯度账
账越大，说明过去动得越猛
以后更新它时就自动收小步子
```

## 4. AdaGrad 的问题也很直观

AdaGrad 的优点来自这本账，问题也来自这本账。

它的账本是这样加的：

$$
s_t=s_{t-1}+g_t^2
$$

注意这个式子里只有“加”，没有“忘”。

所以 $s_t$ 只会越来越大，不会自己变小。

这会带来一个很实际的问题：训练越往后，分母越可能变大。

$$
\sqrt{s_t}+\epsilon
$$

分母越大，实际步子越小：

$$
\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

你可以把它想成：AdaGrad 一开始很聪明，知道给活跃参数减速；但它太记历史了，早期的梯度会一直压着后面的更新。

于是训练后期可能出现这种情况：

```text
模型还没学到理想位置
但很多参数的实际步子已经被压得很小
训练开始变慢，甚至像走不动了一样
```

所以 AdaGrad 适合一些梯度比较稀疏的问题，因为它会照顾那些不常被更新的参数；但在深度神经网络里，如果训练时间较长，它的学习率可能衰减得太厉害。

一句话记：

```text
AdaGrad 会自动调步子，但历史梯度账只加不减，后期容易越走越慢。
```

## 5. RMSProp 为什么出现

RMSProp 可以理解成对 AdaGrad 的一个非常自然的修改。

AdaGrad 的问题是：

```text
过去所有梯度平方都一直累计，太久以前的事情也一直算数。
```

RMSProp 就说：那我们不要把所有历史都永久记住，只看“最近一段时间”的梯度大小趋势。

这就用到了前面学过的指数加权移动平均。

AdaGrad 的记账方式是：

$$
s_t=s_{t-1}+g_t^2
$$

RMSProp 改成：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

这两个式子的区别非常重要。

AdaGrad 是：

```text
旧账 + 新账，一直加
```

RMSProp 是：

```text
旧账打个折 + 新账记一点
```

其中 $\beta$ 通常接近 $1$，比如 $0.9$ 或 $0.99$。它表示：历史信息还要保留多少。

所以 RMSProp 不会让 $s_t$ 无限增大得那么厉害，因为旧梯度的影响会慢慢变淡。

一句话记：

```text
RMSProp = AdaGrad 的账本改良版：不再永久记住所有历史，而是更看重最近的梯度情况。
```

## 6. RMSProp 的公式怎么读

RMSProp 先统计最近梯度平方的大概水平：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

再更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

这两个式子可以按三句话读。

第一句：$g_t$ 决定这一次往哪个方向改。

因为梯度告诉我们损失函数上升最快的方向，所以更新时要往反方向走。

第二句：$s_t$ 记录这个参数最近的梯度大不大。

如果最近梯度经常大，$s_t$ 就大；如果最近梯度比较小，$s_t$ 就小。

第三句：把 $\sqrt{s_t}$ 放在分母里，是为了自动调节实际步子。

如果 $s_t$ 大：

$$
\sqrt{s_t}+\epsilon \text{ 变大}
$$

那么：

$$
\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

就会变小，参数更新会更谨慎。

如果 $s_t$ 小，分母没那么大，更新就不会被压得太狠。

所以 RMSProp 的公式不是凭空来的，它就是沿着这个逻辑走出来的：

1. 先看每个参数最近的梯度大小。
2. 再根据这个大小调整当前更新的步幅。
3. 梯度经常较大，就少走一些；梯度经常较小，就不要把步幅压得太小。

$\epsilon$ 是一个很小的数，主要是防止分母为 $0$，让计算更稳定。

## 7. RMSProp 解决了什么

RMSProp 主要解决两个问题。

第一，它让不同参数有不同的实际学习率。

第二，它不像 AdaGrad 那样让历史梯度平方无限累积，而是更关注近期趋势。

所以 RMSProp 比 AdaGrad 更适合非平稳的深度学习训练过程。

这里的非平稳可以理解为：训练过程中，参数一直在变，损失曲面上的当前位置一直在变，早期梯度统计不应该永远主导后期训练。

## 8. Momentum 和 RMSProp 的区别

Momentum 和 RMSProp 都用到了 EMA，但记住的东西不一样。

Momentum 记住的是梯度本身：

$$
m_t=\beta m_{t-1}+(1-\beta)g_t
$$

它关心的是：最近一段时间大方向往哪里走。

RMSProp 记住的是梯度平方：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

它关心的是：最近一段时间这个参数的梯度大不大。

所以：

```text
Momentum 解决方向抖动
RMSProp 解决不同参数步幅不一样的问题
```

## 9. Adam 是怎么组合二者的

Adam 可以理解成 Momentum 和 RMSProp 的结合。

它既记录梯度的一阶矩，也记录梯度平方的二阶矩。

一阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

二阶矩：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$m_t$ 像 Momentum，表示方向趋势。

$v_t$ 像 RMSProp，表示梯度大小趋势。

Adam 的核心思想是：

- 用 $m_t$ 决定往哪里走。
- 用 $v_t$ 调整每个参数走多大步。

## 10. Adam 为什么需要偏差修正

Adam 的 $m_t$ 和 $v_t$ 通常从 $0$ 开始：

$$
m_0=0,\quad v_0=0
$$

这会让前几步的移动平均偏小。

所以 Adam 会做偏差修正：

$$
\hat{m}_t=\frac{m_t}{1-\beta_1^t}
$$

$$
\hat{v}_t=\frac{v_t}{1-\beta_2^t}
$$

这和上一节指数加权移动平均里的偏差修正是同一个思想。

修正后的 Adam 更新是：

$$
\theta_t=\theta_{t-1}-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

## 11. 为什么 Adam 常用默认参数

Adam 常见默认值是：

$$
\beta_1=0.9
$$

$$
\beta_2=0.999
$$

$\beta_1=0.9$ 表示一阶矩大约参考最近 $10$ 步左右的梯度方向。

因为：

$$
\frac{1}{1-0.9}=10
$$

$\beta_2=0.999$ 表示二阶矩参考更长时间的梯度平方趋势。

因为：

$$
\frac{1}{1-0.999}=1000
$$

直觉是：方向可以稍微灵活一点，但梯度大小的统计希望更稳定。

## 12. 为什么学完 Adam 还要学习 AdamW

Adam 已经解决了两个优化问题：用一阶矩估计方向，用二阶矩为不同参数调整步幅。

但模型训练不只关心训练集损失下降得快不快，还关心模型能不能在新数据上表现良好。参数过大、模型过度依赖训练数据中的细节时，可能出现过拟合。

AdamW 在 Adam 的基础上重点处理 **weight decay（权重衰减）**。名字中的 W 就来自 weight decay。

先记住一句话：

> **Adam 决定怎样利用梯度更新参数；AdamW 在完成 Adam 更新的同时，还独立地让权重缩小一点。**

这也是它经常出现在 Transformer、ViT 和其他大型神经网络训练配置中的原因之一。

## 13. 权重衰减到底在做什么

假设当前参数是 $\theta_{t-1}$，学习率是 $\eta$，权重衰减系数是 $\lambda$。只看权重衰减这一步：

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}
$$

因为 $1-\eta\lambda$ 通常略小于 $1$，所以参数每更新一次，都会向 $0$ 缩小一点。

例如，某个权重为 $2$，如果这一轮的缩放因子为 $0.99$，衰减后就是：

$$
2\times0.99=1.98
$$

它不是把权重直接变成 $0$，而是在训练过程中持续抑制权重无限增大。

所以权重衰减的直觉是：**完成任务学习的同时，不要让模型为了记住训练数据而把参数推得过大。**

## 14. 为什么在 SGD 中，L2 正则化等价于权重衰减

先给数据损失加入 $L_2$ 正则项：

$$
\mathcal{L}_{\mathrm{total}}
=\mathcal{L}_{\mathrm{data}}
+\frac{\lambda}{2}\lVert\theta\rVert_2^2
$$

对参数求梯度：

$$
\nabla_{\theta}\mathcal{L}_{\mathrm{total}}
=\nabla_{\theta}\mathcal{L}_{\mathrm{data}}+\lambda\theta
$$

令数据损失产生的梯度为：

$$
g_t=\nabla_{\theta}\mathcal{L}_{\mathrm{data}}
$$

代入 SGD 更新公式：

$$
\begin{aligned}
\theta_t
&=\theta_{t-1}-\eta(g_t+\lambda\theta_{t-1})\\
&=(1-\eta\lambda)\theta_{t-1}-\eta g_t.
\end{aligned}
$$

现在可以把它拆成两部分：

- $-\eta g_t$：根据任务梯度学习。
- $(1-\eta\lambda)\theta_{t-1}$：让原来的权重缩小。

因此在普通 SGD 中，把 $L_2$ 正则项加到损失里，最终确实会产生与权重衰减相同的更新形式。

## 15. 为什么到了 Adam，这两件事不再等价

如果仍然把 $L_2$ 正则项放进损失函数，Adam 接收到的梯度就会变成：

$$
g_t^{L_2}=g_t+\lambda\theta_{t-1}
$$

Adam 不会像 SGD 那样直接使用这个梯度。它会先把整个 $g_t^{L_2}$ 放进一阶矩和二阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t^{L_2}
$$

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)(g_t^{L_2})^2
$$

然后再做自适应缩放：

$$
\theta_t=\theta_{t-1}
-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

这意味着 $\lambda\theta_{t-1}$ 不仅负责缩小权重，它还参与了 $m_t$ 和 $v_t$ 的统计，并被每个参数不同的自适应分母重新缩放。

结果是：原本想要的“统一缩小权重”，与 Adam 的“按参数调整步幅”缠在了一起。不同参数受到的实际衰减强度可能不同。

这就是 AdamW 要解决的核心问题。

## 16. AdamW 的更新公式是怎么来的

AdamW 先只用数据损失的梯度 $g_t$ 计算 Adam 的一阶矩、二阶矩和自适应更新方向：

$$
u_t=\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

然后把任务更新和权重衰减分开执行：

$$
\theta_t
=\theta_{t-1}-\eta u_t-\eta\lambda\theta_{t-1}
$$

整理后得到：

$$
\theta_t
=(1-\eta\lambda)\theta_{t-1}-\eta u_t
$$

这个式子里有两条互不混淆的路线：

$$
\begin{aligned}
\text{任务学习：}&\quad -\eta u_t,\\
\text{权重衰减：}&\quad -\eta\lambda\theta_{t-1}.
\end{aligned}
$$

关键在于：$\lambda\theta_{t-1}$ 不再进入 Adam 的 $m_t$ 和 $v_t$，而是在 Adam 自适应更新之外单独作用。

这就是 **decoupled weight decay（解耦权重衰减）**。

## 17. Adam 和 AdamW 到底差在哪里

| 对比项 | Adam 配合损失中的 $L_2$ 正则 | AdamW |
|---|---|---|
| Adam 的一阶矩、二阶矩 | 使用 $g_t+\lambda\theta$ 统计 | 只使用数据梯度 $g_t$ 统计 |
| 权重缩小方式 | 与自适应梯度混合 | 在自适应更新之外单独衰减 |
| 每个参数的衰减效果 | 会受到自适应分母影响 | 直接按 $1-\eta\lambda$ 缩小 |
| 核心目的 | 优化与正则化耦合 | 优化与正则化解耦 |

所以 AdamW 不是把 Adam 的一阶矩、二阶矩全部推翻重做。

**AdamW 的 Adam 部分仍然是 Adam，真正改变的是权重衰减放在哪里。**

如果 $\lambda=0$，没有权重衰减，那么 AdamW 就退化成 Adam。

## 18. 为什么 Transformer 和 ViT 经常使用 AdamW

Transformer 和 ViT 通常参数较多，而且包含大量线性投影矩阵、Attention 参数和 FFN 参数。训练这些模型时，我们往往同时希望：

1. 使用 Adam 的自适应步幅，让训练更容易启动。
2. 使用权重衰减抑制参数无约束增大，改善泛化。
3. 不让正则化强度被 Adam 的自适应分母意外改变。

AdamW 正好把这三件事组合起来，所以它成为 Transformer、ViT 等模型中很常见的优化器选择。

但要注意：常见不等于任何任务都一定最好。学习率和权重衰减系数仍然需要结合验证集表现选择。

## 19. 学习率和 weight decay 怎么共同影响更新

AdamW 的衰减项是：

$$
\eta\lambda\theta_{t-1}
$$

因此真正每一步缩小多少，不只由 $\lambda$ 决定，还与学习率 $\eta$ 有关。

- $\eta$：控制整体更新步幅。
- $\lambda$：控制权重衰减强度。
- $\eta\lambda$：决定这一轮实际缩小权重的比例。

$\lambda$ 太小，正则作用可能不明显；$\lambda$ 太大，参数会被压得过强，模型可能欠拟合。

进阶实践中，bias 和 LayerNorm 的缩放、偏移参数经常被设置为不参与权重衰减，因为这些参数与普通权重矩阵的作用不同。这是一种常见训练策略，不是 AdamW 公式本身的强制要求。

## 20. 优化器之间的逻辑关系

可以把这些优化器放在一条线里理解：

$$
\begin{aligned}
\mathrm{SGD}
&\rightarrow \mathrm{Momentum} && \text{记住历史方向，减少抖动}\\
&\rightarrow \mathrm{AdaGrad} && \text{给不同参数调整步幅}\\
&\rightarrow \mathrm{RMSProp} && \text{用 EMA 记录近期梯度平方}\\
&\rightarrow \mathrm{Adam} && \text{同时记方向和梯度大小}\\
&\rightarrow \mathrm{AdamW} && \text{把 Adam 更新与权重衰减解耦}.
\end{aligned}
$$

AdamW 不是突然出现的新优化器思路，而是在 Adam 已有自适应更新的基础上，把正则化路线理顺。

## 21. 本节总结与自检

这一节最重要的结论是：

> **SGD 中，损失里的 $L_2$ 正则化可以推出同样的权重衰减形式；Adam 中，正则梯度会进入一阶矩和二阶矩，因此两者不再等价。AdamW 把权重衰减从自适应梯度中解耦出来。**

AdamW 的核心公式是：

$$
\theta_t
=(1-\eta\lambda)\theta_{t-1}
-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

请尝试回答：

1. AdamW 中的 W 代表什么？
2. 为什么 $L_2$ 正则化在 SGD 中等价于权重衰减？
3. 为什么把 $\lambda\theta$ 直接加入 Adam 的梯度会产生耦合？
4. AdamW 的任务更新和权重衰减分别对应公式中的哪一项？
5. 当 $\lambda=0$ 时，AdamW 会变成什么？

到这里，优化器这条线就比较完整了。下一步可以继续学习学习率调度：为什么训练过程中学习率常常不是固定不变的。